In [81]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import time

torch.manual_seed(0)
np.random.seed(0)

Simulate Data:

In [82]:
GLOBAL_LOGITS = np.random.randn(4, 6)

def simulate_data(N=800, d=4):
    X = np.random.normal(size=(N, d))

    # FIXED logits
    raw = np.exp(X @ GLOBAL_LOGITS)

    # multi-sigmoid structure
    denom = 1 + raw.sum(axis=1, keepdims=True)
    probs = raw / denom
    survival = 1 / denom

    states = []
    times = []

    for i in range(N):
        t = 0
        while True:
            t += 1

            full_probs = np.concatenate(([survival[i, 0]], probs[i]))

            # numerical safety
            full_probs = np.clip(full_probs, 1e-12, 1.0)
            full_probs = full_probs / full_probs.sum()

            event = np.random.choice(7, p=full_probs)

            if event == 0:
                continue
            else:
                states.append(event)
                times.append(t)
                break

    return (
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(times, dtype=torch.float32),
        torch.tensor(states, dtype=torch.long),
        survival,   # NEW
        probs       # NEW
    )

Models: (Constant, Linear, NN)

In [83]:
class BaseModel(nn.Module):
    def __init__(self, input_dim, hidden=50, model_type="nn"):
        super().__init__()
        self.model_type = model_type

        if model_type == "constant":
            self.param = nn.Parameter(torch.zeros(1))

        elif model_type == "linear":
            self.layer = nn.Linear(input_dim, 1)

        else:
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden),
                nn.SiLU(),
                nn.Linear(hidden, hidden),
                nn.SiLU(),
                nn.Linear(hidden, 1)
            )

    def forward(self, x):
        if self.model_type == "constant":
            return self.param.expand(x.shape[0])
        elif self.model_type == "linear":
            return self.layer(x).squeeze()
        else:
            return self.net(x).squeeze()
        
#Multi Sigmoid:
def multi_sigmoid(logits):
    exp_vals = torch.exp(torch.stack(logits, dim=1))
    denom = 1 + exp_vals.sum(dim=1, keepdim=True)
    probs = exp_vals / denom
    survival = 1 / denom
    return probs, survival


Loss and MSE:

In [84]:
def loss_fn(models, X, T, state):
    logits = [m(X) for m in models]
    probs, survival = multi_sigmoid(logits)

    # full probability vector: [survival, p1,...,p6]
    full_probs = torch.cat([survival, probs], dim=1)

    # pick the probability of observed event
    p_obs = full_probs[torch.arange(len(state)), state]

    # discrete-time likelihood:
    # log P(T=t, state=k) = log(p_k) + (t-1)*log(p_survival)
    loglik = torch.log(p_obs) + (T - 1) * torch.log(survival.squeeze())

    return -torch.sum(loglik)

def compute_mse(pred, true):
    return np.mean((pred - true)**2)


Training:

In [85]:
def train_models(models, X, T, state, lr=1e-3, epochs=200, verbose=False):
    # Put all models in training mode
    for m in models:
        m.train()

    # Collect parameters
    params = []
    for m in models:
        params += list(m.parameters())

    optimizer = optim.Adam(params, lr=lr)

    start_time = time.time()
    loss_hist = []

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Compute loss (DISCRETE-TIME likelihood)
        loss = loss_fn(models, X, T, state)

        # Safety check (helps debugging instability)
        if torch.isnan(loss):
            raise ValueError(f"NaN loss at epoch {epoch}")

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)

        optimizer.step()

        loss_hist.append(loss.item())

        # Optional logging
        if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

    total_time = time.time() - start_time

    return models, loss_hist, total_time

Prediction:

In [86]:
def predict(models, X):
    logits = [m(X) for m in models]
    probs, survival = multi_sigmoid(logits)
    return torch.cat([survival, probs], dim=1)  # shape (N,7)

Brier Score:

In [87]:
def state_matrix(state):
    N = len(state)
    mat = np.zeros((N,7))
    for i in range(N):
        mat[i, state[i]] = 1
    return mat


def brier_score(pred, obs):
    return ((pred - obs)**2).sum(axis=1)

Run full loop:

In [88]:
Nruns = 100
results = []

for run in range(Nruns):
    print(f"Run {run+1}/{Nruns}")

    # DATA
    X, T, state, survival, probs = simulate_data()
    X_val, T_val, state_val, survival_val, probs_val = simulate_data(300)

    # TRUE probabilities
    true = np.concatenate([survival_val, probs_val], axis=1)

    # MODELS
    models_const = [BaseModel(4, model_type="constant") for _ in range(6)]
    models_lin = [BaseModel(4, model_type="linear") for _ in range(6)]
    models_nn = [BaseModel(4, model_type="nn") for _ in range(6)]

    # TRAIN
    _, _, t_const = train_models(models_const, X, T, state, lr=1e-2, epochs=100, verbose=False)
    _, _, t_lin = train_models(models_lin, X, T, state, lr=1e-2, epochs=150)
    _, _, t_nn = train_models(models_nn, X, T, state, lr=1e-3, epochs=200)

    # LOSSES (train)
    loss_const = loss_fn(models_const, X, T, state).item()
    loss_lin = loss_fn(models_lin, X, T, state).item()
    loss_nn = loss_fn(models_nn, X, T, state).item()

    # LOSSES (validation)
    loss_const_val = loss_fn(models_const, X_val, T_val, state_val).item()
    loss_lin_val = loss_fn(models_lin, X_val, T_val, state_val).item()
    loss_nn_val = loss_fn(models_nn, X_val, T_val, state_val).item()

    # PREDICTIONS
    pred_const = predict(models_const, X_val).detach().numpy()
    pred_lin = predict(models_lin, X_val).detach().numpy()
    pred_nn = predict(models_nn, X_val).detach().numpy()

    # TRUE (one-hot)
    true = state_matrix(state_val.numpy())

    # =========================
    # MSE COMPUTATION
    # =========================

    # CONSTANT
    MSE11_const = compute_mse(pred_const[:,0], true[:,0])
    MSE12_const = compute_mse(pred_const[:,1], true[:,1])
    MSE13_const = compute_mse(pred_const[:,2], true[:,2])

    # LINEAR
    MSE11_lin = compute_mse(pred_lin[:,0], true[:,0])
    MSE12_lin = compute_mse(pred_lin[:,1], true[:,1])
    MSE13_lin = compute_mse(pred_lin[:,2], true[:,2])

    # NN
    MSE11_nn = compute_mse(pred_nn[:,0], true[:,0])
    MSE12_nn = compute_mse(pred_nn[:,1], true[:,1])
    MSE13_nn = compute_mse(pred_nn[:,2], true[:,2])

    # =========================
    # BRIER SCORE
    # =========================
    BS_const = brier_score(pred_const, true).mean()
    BS_lin = brier_score(pred_lin, true).mean()
    BS_nn = brier_score(pred_nn, true).mean()

    # SAVE RESULTS
    results.append([
        t_const, t_lin, t_nn,
        loss_const, loss_lin, loss_nn,
        loss_const_val, loss_lin_val, loss_nn_val,
        BS_const, BS_lin, BS_nn,
        MSE11_const, MSE12_const, MSE13_const,
        MSE11_lin, MSE12_lin, MSE13_lin,
        MSE11_nn, MSE12_nn, MSE13_nn
    ])

Run 1/100
Run 2/100
Run 3/100
Run 4/100
Run 5/100
Run 6/100
Run 7/100
Run 8/100
Run 9/100
Run 10/100
Run 11/100
Run 12/100
Run 13/100
Run 14/100
Run 15/100
Run 16/100
Run 17/100
Run 18/100
Run 19/100
Run 20/100
Run 21/100
Run 22/100
Run 23/100
Run 24/100
Run 25/100
Run 26/100
Run 27/100
Run 28/100
Run 29/100
Run 30/100
Run 31/100
Run 32/100
Run 33/100
Run 34/100
Run 35/100
Run 36/100
Run 37/100
Run 38/100
Run 39/100
Run 40/100
Run 41/100
Run 42/100
Run 43/100
Run 44/100
Run 45/100
Run 46/100
Run 47/100
Run 48/100
Run 49/100
Run 50/100
Run 51/100
Run 52/100
Run 53/100
Run 54/100
Run 55/100
Run 56/100
Run 57/100
Run 58/100
Run 59/100
Run 60/100
Run 61/100
Run 62/100
Run 63/100
Run 64/100
Run 65/100
Run 66/100
Run 67/100
Run 68/100
Run 69/100
Run 70/100
Run 71/100
Run 72/100
Run 73/100
Run 74/100
Run 75/100
Run 76/100
Run 77/100
Run 78/100
Run 79/100
Run 80/100
Run 81/100
Run 82/100
Run 83/100
Run 84/100
Run 85/100
Run 86/100
Run 87/100
Run 88/100
Run 89/100
Run 90/100
Run 91/100
Run 92/1

Save into "python_performance_metrics.csv" to be compared with julia output

In [89]:
columns = [
    "time_const","time_lin","time_NN",
    "loss_const","loss_lin","loss_NN",
    "loss_const_val","loss_lin_val","loss_NN_val",
    "BS_const","BS_lin","BS_NN",
    "MSE11_const_val","MSE12_const_val","MSE13_const_val",
    "MSE11_lin_val","MSE12_lin_val","MSE13_lin_val",
    "MSE11_NN_val","MSE12_NN_val","MSE13_NN_val"
]

df = pd.DataFrame(results, columns=columns)
df.to_csv("python_performance_metrics.csv", index=False)

print(df.head())

   time_const  time_lin   time_NN   loss_const     loss_lin      loss_NN  \
0    0.087498  0.221129  1.108962  1741.681641  1159.692261  1069.937744   
1    0.096667  0.196498  1.057058  1725.491211  1182.431885  1055.682983   
2    0.085501  0.208619  1.070024  1731.560303  1218.152710  1115.633057   
3    0.086501  0.204123  1.012492  1751.807129  1224.799927  1099.540283   
4    0.086503  0.216947  1.025545  1703.958130  1155.453369  1029.633789   

   loss_const_val  loss_lin_val  loss_NN_val  BS_const  ...     BS_NN  \
0      669.083130    453.381073   483.513458  0.835604  ...  0.613221   
1      718.225098    484.954315   524.631470  0.839587  ...  0.623728   
2      686.279480    463.557129   478.739166  0.832715  ...  0.592207   
3      675.835693    504.397614   541.131104  0.838930  ...  0.675272   
4      663.010010    451.416138   474.760681  0.828657  ...  0.624900   

   MSE11_const_val  MSE12_const_val  MSE13_const_val  MSE11_lin_val  \
0         0.015386         0.1008

Transition Matrix:

In [ ]:
Nruns = 100
estimated_matrices = []

for run in range(Nruns):
    print(f"Run {run+1}/{Nruns}")

    # DATA
    X, T, state, survival, probs = simulate_data()

    true_full = np.concatenate([survival, probs], axis=1)

    # MODEL
    models_nn = [BaseModel(4, model_type="nn") for _ in range(6)]
    train_models(models_nn, X, T, state, lr=1e-3, epochs=200)

    # PREDICT
    pred = predict(models_nn, X).detach().numpy()

    # AVERAGE
    estimated_mean = pred.mean(axis=0)
    estimated_matrices.append(estimated_mean)

# FINAL ESTIMATE
estimated_matrix = np.mean(estimated_matrices, axis=0)

# TRUE (single large dataset)
X_big = np.random.normal(size=(5000, 4))
raw = np.exp(X_big @ GLOBAL_LOGITS)
denom = 1 + raw.sum(axis=1, keepdims=True)

true_matrix = np.mean(
    np.concatenate([1/denom, raw/denom], axis=1),
    axis=0
)

# ERROR
error_matrix = np.abs(estimated_matrix - true_matrix)
# DISPLAY
import matplotlib.pyplot as plt
import numpy as np

#Side by side bar chart
labels = ["Survival", "1", "2", "3", "4", "5", "6"]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10,6))

plt.bar(x - width/2, true_matrix, width, label='True')
plt.bar(x + width/2, estimated_matrix, width, label='Estimated')

plt.xticks(x, labels)
plt.ylabel("Probability")
plt.title("True vs Estimated Transition Probabilities")
plt.legend()


# Error bar plot
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))

plt.bar(labels, error_matrix)

plt.ylabel("Absolute Error")
plt.title("Estimation Error per Transition")

plt.tight_layout()
plt.show()



#heat map
plt.figure(figsize=(10,2))

plt.imshow([estimated_matrix], aspect='auto')
plt.colorbar()
plt.xticks(range(7), labels)
plt.yticks([])
plt.title("Estimated Transition Probabilities")

plt.show()

#Table
df = pd.DataFrame({
    "State": labels,
    "True": true_matrix,
    "Estimated": estimated_matrix,
    "Absolute Error": error_matrix
})

print(df.round(4))

Run 1/100
Run 2/100
Run 3/100
Run 4/100
Run 5/100
Run 6/100
Run 7/100
Run 8/100
Run 9/100
Run 10/100
Run 11/100
Run 12/100
Run 13/100
Run 14/100
Run 15/100
Run 16/100
Run 17/100
Run 18/100
Run 19/100
Run 20/100
Run 21/100
Run 22/100
Run 23/100
Run 24/100
Run 25/100
Run 26/100
Run 27/100
Run 28/100
Run 29/100
Run 30/100
Run 31/100
Run 32/100
Run 33/100
Run 34/100
Run 35/100
Run 36/100
Run 37/100
Run 38/100
Run 39/100
Run 40/100
Run 41/100
Run 42/100
Run 43/100
Run 44/100
Run 45/100
Run 46/100
Run 47/100
Run 48/100
Run 49/100
Run 50/100
Run 51/100
Run 52/100
Run 53/100
Run 54/100
Run 55/100
Run 56/100
Run 57/100
Run 58/100
Run 59/100
Run 60/100
Run 61/100
Run 62/100
Run 63/100
Run 64/100
Run 65/100
Run 66/100
Run 67/100
Run 68/100
Run 69/100
Run 70/100
Run 71/100
Run 72/100
Run 73/100
Run 74/100
Run 75/100
Run 76/100
Run 77/100
Run 78/100
Run 79/100
Run 80/100
Run 81/100
Run 82/100
Run 83/100
Run 84/100
Run 85/100
Run 86/100
Run 87/100
Run 88/100
Run 89/100
Run 90/100
Run 91/100
Run 92/1